In [ ]:
from __future__ import annotations
 
import sys
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
BASE_DIR = Path("../data/raw")

datasets = {
    "channels": [],
}

for csv_file in BASE_DIR.rglob("*.csv"):
    try:
        df = pd.read_csv(csv_file)

        # df["source_file"] = csv_file.name
        # df["source_path"] = str(csv_file)

        id_dir = next(
            (part for part in csv_file.parts if part.startswith("id_")),
            None
        )
        df["collection_id"] = id_dir

        stem_parts = csv_file.stem.split("_")
        country = stem_parts[-1] if len(stem_parts) > 2 else None
        df["country"] = country

        filename = csv_file.name.lower()


        if "channel" in filename:
            datasets["channels"].append(df)

    except Exception as e:
        print(f"Erro ao ler {csv_file}: {e}")


channels_df = pd.concat(datasets["channels"], ignore_index=True) \
    if datasets["channels"] else pd.DataFrame()

print(f"Channels: {len(channels_df)}")


In [ ]:
from sklearn.impute import SimpleImputer

df = channels_df.copy()
df = df.drop_duplicates(subset="channel_id")
text_cols = ["channel_id", "title", "description", "custom_url", "country"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].replace(["", "None", "nan", "NaN", "unknown", "UNKNOWN"], np.nan)

if "published_at" in df.columns:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")

num_cols = ["view_count", "subscriber_count", "video_count", "hidden_subscriber_count"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "country" in df.columns:
    df["country"] = (
        df["country"]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace({"": np.nan})
    )

if "custom_url" in df.columns:
    df["has_custom_url"] = df["custom_url"].notna().astype(int)
else:
    df["has_custom_url"] = 0

if "hidden_subscriber_count" in df.columns:
    df["hidden_subscriber_count"] = (
        df["hidden_subscriber_count"]
        .fillna(0)
        .astype(int)
        .clip(0, 1)
    )
else:
    df["hidden_subscriber_count"] = 0

for col in ["view_count", "subscriber_count", "video_count"]:
    if col in df.columns:
        df[f"{col}_missing"] = df[col].isna().astype(int)

imputer = SimpleImputer(strategy="median")
cols_to_impute = [c for c in ["view_count", "subscriber_count", "video_count"] if c in df.columns]
if cols_to_impute:
    df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

if "view_count" in df.columns:
    df["view_count_log"] = np.log1p(df["view_count"])
if "subscriber_count" in df.columns:
    df["subscriber_count_log"] = np.log1p(df["subscriber_count"])
if "video_count" in df.columns:
    df["video_count_log"] = np.log1p(df["video_count"])

if set(["subscriber_count", "video_count"]).issubset(df.columns):
    df["subscribers_per_video"] = np.where(
        df["video_count"] > 0,
        df["subscriber_count"] / df["video_count"],
        np.nan
    )

if set(["view_count", "video_count"]).issubset(df.columns):
    df["views_per_video"] = np.where(
        df["video_count"] > 0,
        df["view_count"] / df["video_count"],
        np.nan
    )

if set(["subscriber_count", "view_count"]).issubset(df.columns):
    df["subscribers_per_view"] = np.where(
        df["view_count"] > 0,
        df["subscriber_count"] / df["view_count"],
        np.nan
    )

if "published_at" in df.columns:
    df["channel_year"] = df["published_at"].dt.year
    df["channel_month"] = df["published_at"].dt.month
channel_clean = df[["channel_id","subscriber_count", "view_count", "video_count"]]
channel_clean


In [ ]:
from pathlib import Path

BASE_DIR = Path("../data/processed")
for csv_file in BASE_DIR.rglob("processed_videos_20260601_153039.csv"):
    
# # Scan the current working directory
# for item in Path('.').iterdir():
#     if item.is_file():
#         print(item.name)
    videos_df = pd.read_csv(csv_file)
videos_df.head()

In [ ]:
videos = videos_df.copy()
channels = channel_clean.copy()

if "published_at" in videos.columns:
    videos["published_at"] = pd.to_datetime(videos["published_at"], errors="coerce")

if "published_at" in channels.columns:
    channels = channels.rename(columns={"published_at": "channel_published_at"})

merged = videos.merge(
    channels,
    on="channel_id",
    how="left",
    suffixes=("", "_channel")
)

if set(["published_at", "channel_published_at"]).issubset(merged.columns):
    merged["channel_age_days_at_video"] = (
        merged["published_at"] - merged["channel_published_at"]
    ).dt.days

    merged["channel_age_days_at_video"] = merged["channel_age_days_at_video"].clip(lower=0)

    merged["channel_age_years_at_video"] = merged["channel_age_days_at_video"] / 365.25

if "channel_age_days_at_video" in merged.columns:
    merged["channel_age_days_at_video_missing"] = merged["channel_age_days_at_video"].isna().astype(int)

merged

In [ ]:
merged.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = {
    "bg":      "#0D0F1A",
    "panel":   "#141726",
    "accent1": "#FF4D6D",
    "accent2": "#4CC9F0",
    "accent3": "#F4A261",
    "accent4": "#2EC4B6",
    "text":    "#E8EAF0",
    "muted":   "#6C7293",
}

def _setup_dark_theme():
    plt.rcParams.update({
        "figure.facecolor": PALETTE["bg"],
        "axes.facecolor": PALETTE["panel"],
        "axes.edgecolor": PALETTE["muted"],
        "axes.labelcolor": PALETTE["text"],
        "text.color": PALETTE["text"],
        "xtick.color": PALETTE["text"],
        "ytick.color": PALETTE["text"],
        "grid.color": PALETTE["muted"],
        "grid.alpha": 0.22,
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "font.size": 10,
        "legend.facecolor": PALETTE["panel"],
        "legend.edgecolor": PALETTE["muted"],
    })

def _finish(ax, title, xlabel=None, ylabel=None):
    ax.set_title(title, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_alpha(0.5)

def plot_youtube_eda(df: pd.DataFrame, save_dir: str = "plots_youtube", show: bool = True):
    _setup_dark_theme()
    out = Path(save_dir)
    out.mkdir(parents=True, exist_ok=True)

    d = df.copy()

    if "published_at" in d.columns:
        d["published_at"] = pd.to_datetime(d["published_at"], errors="coerce")

    numeric_cols = [
        "category_id", "view_count", "like_count", "comment_count",
        "engagement_rate", "like_ratio", "comment_ratio", "year", "month",
        "view_count_capped", "subscriber_count", "view_count_channel", "video_count"
    ]
    for col in numeric_cols:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    dist_cols = [c for c in [
        "view_count", "like_count", "comment_count", "engagement_rate",
        "like_ratio", "comment_ratio", "subscriber_count",
        "view_count_channel", "video_count", "view_count_capped"
    ] if c in d.columns]

    if dist_cols:
        n = len(dist_cols)
        fig, axes = plt.subplots(n, 1, figsize=(12, 3.6 * n), constrained_layout=True)
        if n == 1:
            axes = [axes]

        colors = [PALETTE["accent1"], PALETTE["accent2"], PALETTE["accent3"], PALETTE["accent4"]]

        for i, col in enumerate(dist_cols):
            ax = axes[i]
            x = d[col].dropna()

            if col in ["view_count", "like_count", "comment_count", "subscriber_count", "view_count_channel", "video_count", "view_count_capped"]:
                x = np.log1p(x[x >= 0])

                ax.hist(x, bins=40, color=colors[i % len(colors)], alpha=0.9)
                _finish(ax, f"Distribuição log1p de {col}", xlabel=f"log1p({col})", ylabel="Frequência")
            else:
                ax.hist(x, bins=40, color=colors[i % len(colors)], alpha=0.9)
                _finish(ax, f"Distribuição de {col}", xlabel=col, ylabel="Frequência")

        fig.suptitle("Distribuições das variáveis principais", y=1.01, fontsize=16)
        fig.savefig(out / "01_distribuicoes_principais.png", dpi=160, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)

    # if "published_at" in d.columns:
    #     tmp = d.dropna(subset=["published_at"]).copy()
    #     if not tmp.empty:
    #         tmp["date"] = tmp["published_at"].dt.to_period("M").dt.to_timestamp()
    #         monthly = tmp.groupby("date").agg(
    #             videos=("video_id", "count") if "video_id" in tmp.columns else ("published_at", "count"),
    #             views=("view_count", "mean") if "view_count" in tmp.columns else ("published_at", "count"),
    #             engagement=("engagement_rate", "mean") if "engagement_rate" in tmp.columns else ("published_at", "count"),
    #         ).reset_index()

    #         fig, axes = plt.subplots(3, 1, figsize=(13, 11), constrained_layout=True)
    #         axes = np.ravel(axes)

    #         if "videos" in monthly:
    #             axes[0].plot(monthly["date"], monthly["videos"], linewidth=2, color=PALETTE["accent2"])
    #             _finish(axes[0], "Vídeos publicados por mês", xlabel="Mês", ylabel="Qtd. de vídeos")

    #         if "views" in monthly:
    #             axes[1].plot(monthly["date"], np.log1p(monthly["views"]), linewidth=2, color=PALETTE["accent1"])
    #             _finish(axes[1], "Média de views por mês (log1p)", xlabel="Mês", ylabel="log1p(média views)")

    #         if "engagement" in monthly:
    #             axes[2].plot(monthly["date"], monthly["engagement"], linewidth=2, color=PALETTE["accent4"])
    #             _finish(axes[2], "Engajamento médio por mês", xlabel="Mês", ylabel="Engagement rate")

    #         fig.savefig(out / "02_series_temporal.png", dpi=160, bbox_inches="tight")
    #         if show:
    #             plt.show()
    #         plt.close(fig)

    if "category_id" in d.columns:
        cat = d["category_id"].dropna().astype(int).value_counts().head(15)
        if not cat.empty:
            fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
            ax.bar(cat.index.astype(str), cat.values, color=PALETTE["accent3"], alpha=0.95)
            _finish(ax, "Top 15 category_id por frequência", xlabel="category_id", ylabel="Qtd. vídeos")
            plt.xticks(rotation=45, ha="right")
            fig.savefig(out / "03_top_categorias.png", dpi=160, bbox_inches="tight")
            if show:
                plt.show()
            plt.close(fig)

    if "country" in d.columns:
        countries = d["country"].astype("string").fillna("NA").value_counts().head(15)
        if not countries.empty:
            fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
            ax.bar(countries.index.astype(str), countries.values, color=PALETTE["accent4"], alpha=0.95)
            _finish(ax, "Top países por volume de vídeos", xlabel="País", ylabel="Qtd. vídeos")
            plt.xticks(rotation=45, ha="right")
            fig.savefig(out / "04_top_paises.png", dpi=160, bbox_inches="tight")
            if show:
                plt.show()
            plt.close(fig)

    pair_cols = [c for c in ["view_count", "like_count", "comment_count", "subscriber_count", "video_count"] if c in d.columns]
    if len(pair_cols) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

        if "view_count" in d.columns and "like_count" in d.columns:
            x = np.log1p(d["view_count"].clip(lower=0))
            y = np.log1p(d["like_count"].clip(lower=0))
            axes[0].scatter(x, y, s=14, alpha=0.35, color=PALETTE["accent2"])
            _finish(axes[0], "views vs likes (log1p)", xlabel="log1p(view_count)", ylabel="log1p(like_count)")

        if "view_count" in d.columns and "comment_count" in d.columns:
            x = np.log1p(d["view_count"].clip(lower=0))
            y = np.log1p(d["comment_count"].clip(lower=0))
            axes[1].scatter(x, y, s=14, alpha=0.35, color=PALETTE["accent1"])
            _finish(axes[1], "views vs comments (log1p)", xlabel="log1p(view_count)", ylabel="log1p(comment_count)")

        fig.savefig(out / "05_relacoes_views_engajamento.png", dpi=160, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)

    corr_candidates = [c for c in [
        "view_count", "like_count", "comment_count", "engagement_rate",
        "like_ratio", "comment_ratio", "subscriber_count",
        "view_count_channel", "video_count", "view_count_capped",
        "year", "month"
    ] if c in d.columns]

    if len(corr_candidates) >= 2:
        corr = d[corr_candidates].corr(numeric_only=True)
        fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
        im = ax.imshow(corr, interpolation="nearest")
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.index)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right")
        ax.set_yticklabels(corr.index)
        _finish(ax, "Matriz de correlação", xlabel="", ylabel="")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(colors=PALETTE["text"])
        fig.savefig(out / "06_correlacao.png", dpi=160, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)

    # if "month" in d.columns and "view_count" in d.columns:
    #     tmp = d[["month", "view_count"]].dropna().copy()
    #     if not tmp.empty:
    #         tmp["month"] = tmp["month"].astype(int)
    #         fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
    #         data = [np.log1p(tmp.loc[tmp["month"] == m, "view_count"].clip(lower=0)) for m in range(1, 13)]
    #         ax.boxplot(data, labels=[str(m) for m in range(1, 13)], showfliers=False)
    #         _finish(ax, "Distribuição de views por mês (log1p)", xlabel="Mês", ylabel="log1p(view_count)")
    #         fig.savefig(out / "07_boxplot_views_mes.png", dpi=160, bbox_inches="tight")
    #         if show:
    #             plt.show()
    #         plt.close(fig)

    print(f"Plots salvos em: {out.resolve()}")

In [ ]:
plot_youtube_eda(merged, save_dir="plots_youtube", show=True)